# Return-contribution attribution

Single-period **return-contribution attribution** decomposes a portfolio's realized return into the pieces that produced it: per-instrument contributions, arbitrary group buckets (sector, strategy, ...), factor contributions, and a Brinson-Fachler split versus a benchmark.

Two functions in `finstack_quant.attribution` drive the workflow, both JSON-in / JSON-out:

- `attribute_return_contribution(spec_json)` — run the attribution and return the result as JSON.
- `validate_return_contribution_json(spec_json)` — check a specification without computing it (raises `ValueError` on malformed input).

Everything is derived from weights and returns, so no pricing or market data is required and the notebook is self-contained.

## What it computes

Each position contributes `weight × return` to the portfolio return. The result groups those contributions four ways:

| Output | Meaning |
|---|---|
| `portfolio_return` | Sum of all instrument contributions (compensated summation). |
| `instrument_contribution` | Per-position `weight`, `return`, `contribution`, and `active_contribution` versus the benchmark. |
| `group_contribution` | Contributions summed within each label of every group dimension. |
| `factor_contribution` | `exposure × factor_return` for each supplied factor. |
| `benchmark_relative` | Brinson-Fachler allocation, selection, and interaction effects (when benchmark fields are present). |

Position weights come from one of three input shapes: market values normalized **gross** (`|mv| / Σ|mv|`), market values normalized **net** (`mv / Σ mv`), or **explicit weights** supplied directly.

## Imports

In [1]:
import json

import pandas as pd

from finstack_quant.attribution import (
    attribute_return_contribution,
    validate_return_contribution_json,
)

print("Imported return-contribution helpers from finstack_quant.attribution.")

Imported return-contribution helpers from finstack_quant.attribution.


## Build a portfolio spec

A specification carries an `as_of` label, a `weighting` mode, the `positions`, and optional `factors`. Each position supplies exactly one of `market_value` or `weight`, a period `return`, optional `groups`, and optional `benchmark_weight` / `benchmark_return` (present on every position or none).

The book below is a five-name long-only equity portfolio (market values total 100,000) with `sector` and `strategy` labels, benchmark weights and returns, and three style factors.

In [2]:
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/return_contribution.json").read_text())

spec = _NOTEBOOK_DATA['spec']

gross_mv = sum(abs(p["market_value"]) for p in spec["positions"])
print(f"{len(spec['positions'])} positions, gross market value = {gross_mv:,.0f}")

5 positions, gross market value = 100,000


## Validate and run

`validate_return_contribution_json` raises `ValueError` when the spec is malformed; otherwise `attribute_return_contribution` returns the result as a JSON string, which we parse back into a dict.

In [3]:
spec_json = json.dumps(spec)
validate_return_contribution_json(spec_json)  # raises ValueError if malformed

result = json.loads(attribute_return_contribution(spec_json))

print(f"as_of:            {spec['as_of']}")
print(f"weighting:        {spec['weighting']}")
print(f"portfolio_return: {result['portfolio_return']:.6f}")

as_of:            2026-03-31
weighting:        gross
portfolio_return: 0.009350


## Instrument contributions

Rows are sorted by instrument id. `contribution = weight × return`, and `active_contribution = contribution − benchmark_weight × benchmark_return`. The instrument contributions sum to `portfolio_return`.

In [4]:
instruments = pd.DataFrame(result["instrument_contribution"])
print(f"sum of contributions: {instruments['contribution'].sum():.6f}")
instruments.round(6)

sum of contributions: 0.009350


,id,weight,return,contribution,active_contribution
0,AAPL.XNAS,0.30,0.018,0.00540,0.00165
1,JPM.XNYS,0.18,0.009,0.00162,0.00022
2,MSFT.XNAS,0.25,0.011,0.00275,0.00011
3,PG.XNYS,0.12,0.004,0.00048,-0.00027
4,XOM.XNYS,0.15,-0.006,-0.00090,-0.00018


## Group contributions

Every distinct key in `groups` becomes its own dimension, and contributions are summed per label within each dimension. A position missing a label for a dimension falls into the `unknown` bucket — visible here for the `strategy` dimension, which `XOM.XNYS` does not set.

In [5]:
group_rows = [
    {"dimension": dimension, "key": row["key"], "contribution": row["contribution"]}
    for dimension, rows in result["group_contribution"].items()
    for row in rows
]
groups = pd.DataFrame(group_rows)
groups.round(6)

,dimension,key,contribution
0,sector,energy,-0.00090
1,sector,financials,0.00162
2,sector,staples,0.00048
3,sector,tech,0.00815
4,strategy,core,0.00863
5,strategy,tactical,0.00162
6,strategy,unknown,-0.00090


## Factor contributions

Each factor contributes `exposure × factor_return`. Factor contributions are reported alongside, and independently of, the holdings decomposition, and are sorted by factor name.

In [6]:
factors = pd.DataFrame(result["factor_contribution"])
factors.round(6)

,factor,exposure,factor_return,contribution
0,momentum,-0.10,0.020,-0.0020
1,quality,0.20,0.008,0.0016
2,value,0.35,0.012,0.0042


## Benchmark-relative attribution (Brinson-Fachler)

When every position carries `benchmark_weight` and `benchmark_return`, the result adds a Brinson-Fachler decomposition of active return. The split uses the `sector` dimension when present, otherwise the first group dimension. By construction `active_return = allocation + selection + interaction`, up to a tiny numerical residual.

Benchmark-relative attribution requires both portfolio and benchmark weights to sum to 1.0.

In [7]:
relative = result["benchmark_relative"]
reconstructed = (
    relative["allocation_effect"]
    + relative["selection_effect"]
    + relative["interaction_effect"]
)

print(pd.Series(relative).round(8).to_string())
print()
print(f"active_return                 : {relative['active_return']:.8f}")
print(f"allocation+selection+interact : {reconstructed:.8f}")
print(f"residual                      : {relative['residual']:.2e}")

benchmark_return      0.007820
active_return         0.001530
allocation_effect     0.000918
selection_effect      0.000465
interaction_effect    0.000148
residual              0.000000

active_return                 : 0.00153000
allocation+selection+interact : 0.00153000
residual                      : 1.63e-19


## Weighting modes: gross vs net market value

For market-value inputs, the `weighting` mode sets how weights are normalized. The two modes agree for a long-only book but diverge once a position is short. `gross` (`|mv| / Σ|mv|`) yields non-negative exposure shares; `net_market_value` (`mv / Σ mv`) yields signed weights, so a short's contribution flips sign relative to its raw return — the natural choice for long/short P&L.

In [8]:
long_short = [
    {"id": "LONG_A", "market_value": 60000.0, "return": 0.020, "groups": {"side": "long"}},
    {"id": "LONG_B", "market_value": 40000.0, "return": 0.010, "groups": {"side": "long"}},
    {"id": "SHORT_C", "market_value": -30000.0, "return": -0.030, "groups": {"side": "short"}},
]


def attribute(weighting):
    payload = {"as_of": "2026-03-31", "weighting": weighting, "positions": long_short}
    return json.loads(attribute_return_contribution(json.dumps(payload)))


runs = {mode: attribute(mode) for mode in ("gross", "net_market_value")}
for mode, res_mode in runs.items():
    print(f"portfolio_return ({mode}): {res_mode['portfolio_return']:.6f}")

comparison = pd.concat(
    {
        mode: pd.DataFrame(res_mode["instrument_contribution"])
        .set_index("id")[["weight", "contribution"]]
        for mode, res_mode in runs.items()
    },
    axis=1,
)
comparison.round(6)

portfolio_return (gross): 0.005385
portfolio_return (net_market_value): 0.035714


gross              net_market_value             
           weight contribution           weight contribution
id                                                          
LONG_A   0.461538     0.009231         0.857143     0.017143
LONG_B   0.307692     0.003077         0.571429     0.005714
SHORT_C  0.230769    -0.006923        -0.428571     0.012857

## Explicit weights

Supply `weight` directly instead of `market_value` when the weights are already known (for example a model or an overlay). Explicit weights need not sum to 1.0 unless you request benchmark-relative attribution.

In [9]:
overlay_spec = {
    "as_of": "2026-03-31",
    "positions": [
        {"id": "A", "weight": 0.5, "return": 0.020, "groups": {"sleeve": "alpha"}},
        {"id": "B", "weight": 0.3, "return": 0.010, "groups": {"sleeve": "alpha"}},
        {"id": "C", "weight": 0.2, "return": -0.010, "groups": {"sleeve": "hedge"}},
    ],
}
overlay = json.loads(attribute_return_contribution(json.dumps(overlay_spec)))
print(f"portfolio_return: {overlay['portfolio_return']:.6f}")
pd.DataFrame(overlay["instrument_contribution"]).round(6)

portfolio_return: 0.011000


,id,weight,return,contribution
0,A,0.5,0.02,0.010
1,B,0.3,0.01,0.003
2,C,0.2,-0.01,-0.002


## Validation and error handling

`validate_return_contribution_json` enforces the spec invariants up front, raising `ValueError` with a descriptive message. The same checks run inside `attribute_return_contribution`, so a spec that validates will also execute.

In [10]:
def check(label, candidate):
    try:
        validate_return_contribution_json(json.dumps(candidate))
        print(f"OK       | {label}")
    except ValueError as exc:
        print(f"rejected | {label}\n           {exc}")


check(
    "mixes market_value and weight on one position",
    {"as_of": "2026-03-31",
     "positions": [{"id": "A", "market_value": 1.0, "weight": 0.5, "return": 0.01}]},
)
check(
    "benchmark fields on only some positions",
    {"as_of": "2026-03-31", "positions": [
        {"id": "A", "weight": 0.5, "return": 0.01, "benchmark_weight": 0.5, "benchmark_return": 0.01},
        {"id": "B", "weight": 0.5, "return": 0.02},
    ]},
)
check(
    "benchmark weights do not sum to 1.0",
    {"as_of": "2026-03-31", "positions": [
        {"id": "A", "weight": 0.5, "return": 0.01, "benchmark_weight": 0.3, "benchmark_return": 0.01},
        {"id": "B", "weight": 0.5, "return": 0.02, "benchmark_weight": 0.3, "benchmark_return": 0.02},
    ]},
)

rejected | mixes market_value and weight on one position
           Validation error: return contribution position 'A' must supply exactly one of market_value or weight
rejected | benchmark fields on only some positions
           Validation error: return contribution benchmark fields must be present on every position or none
rejected | benchmark weights do not sum to 1.0
           Validation error: return contribution benchmark weights must sum to 1.0 for benchmark-relative attribution (got 0.6)


## Takeaways

- `attribute_return_contribution(spec_json)` decomposes a single-period portfolio return into instrument, group, factor, and benchmark-relative pieces; `validate_return_contribution_json(spec_json)` checks a spec without running it.
- Per-instrument `contribution = weight × return` sums exactly to `portfolio_return`; the group buckets and Brinson-Fachler effects are consistent slices of that same total.
- Choose `weighting`: `gross` for exposure-share decomposition, `net_market_value` for signed long/short P&L, or pass explicit `weight`s.
- Supplying `benchmark_weight` / `benchmark_return` on every position unlocks the Brinson-Fachler allocation / selection / interaction split, with `active_return` reconciling to their sum.
- Both functions are JSON-in / JSON-out, so specs and results log, replay, and cross the Python / Rust / WASM boundary unchanged.